# 04 — H&M ALS Recommender

This notebook trains an implicit ALS recommender using the processed H&M interaction data, evaluates it with the same metrics as the popularity baseline, saves model artifacts, and creates demo recommendation files.

In [1]:
import os
import sys
import json
import math
import time
import pickle
import platform
import subprocess
from pathlib import Path
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.12.12
Pandas: 2.3.3
NumPy: 2.0.2


## 1. Install / import ALS dependency

In [2]:
try:
    from implicit.als import AlternatingLeastSquares
    import implicit
    print("implicit:", implicit.__version__)
except Exception:
    print("implicit is not installed. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "implicit", "-q"])
    from implicit.als import AlternatingLeastSquares
    import implicit
    print("implicit:", implicit.__version__)

implicit is not installed. Installing now...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 54.1 MB/s eta 0:00:00
implicit: 0.7.3


## 2. Environment and paths

In [3]:
def detect_environment() -> str:
    if (
        os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
        or Path("/kaggle/input").exists()
        or Path("/kaggle/working").exists()
    ):
        return "kaggle"

    if os.environ.get("COLAB_RELEASE_TAG") is not None:
        return "colab"

    return "local"


ENVIRONMENT = detect_environment()
IN_COLAB = ENVIRONMENT == "colab"
IN_KAGGLE = ENVIRONMENT == "kaggle"

print("Detected environment:", ENVIRONMENT)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Google Drive mount because this is not real Colab.")

Detected environment: kaggle
Skipping Google Drive mount because this is not real Colab.


In [4]:
PROJECT_NAME = "hm-recommender"


def find_processed_data_dir() -> Path:
    expected_file = "train_interactions_aggregated.parquet"

    candidate_dirs = [
        Path("/kaggle/working") / PROJECT_NAME / "data" / "processed",
        Path("/content/drive/MyDrive") / PROJECT_NAME / "data" / "processed",
        Path.cwd() / "data" / "processed",
        Path.cwd() / PROJECT_NAME / "data" / "processed",
    ]

    for candidate in candidate_dirs:
        if (candidate / expected_file).exists():
            return candidate

    if Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").rglob(expected_file))
        if matches:
            return matches[0].parent

    raise FileNotFoundError(
        "Could not find processed data folder. Run 01-hm-preprocessing-fixed.ipynb first "
        "or attach its output dataset."
    )


PROCESSED_DATA_DIR = find_processed_data_dir()

if IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
elif IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd()

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ALS_ARTIFACTS_DIR = ARTIFACTS_DIR / "als_recommender"
BASELINE_ARTIFACTS_DIR = ARTIFACTS_DIR / "popularity_baseline"

for folder in [ARTIFACTS_DIR, ALS_ARTIFACTS_DIR, BASELINE_ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("ALS_ARTIFACTS_DIR:", ALS_ARTIFACTS_DIR)

PROJECT_DIR: /kaggle/working/hm-recommender
PROCESSED_DATA_DIR: /kaggle/input/datasets/albaky7/hm-recommender/hm-recommender/data/processed
ALS_ARTIFACTS_DIR: /kaggle/working/hm-recommender/artifacts/als_recommender


In [5]:
required_processed_files = [
    "train_interactions.parquet",
    "val_interactions.parquet",
    "test_interactions.parquet",
    "train_interactions_aggregated.parquet",
    "articles_processed.parquet",
    "article_id_map.parquet",
    "customer_id_map.parquet",
    "sample_submission_processed.parquet",
]

missing = []
for filename in required_processed_files:
    path = PROCESSED_DATA_DIR / filename
    if path.exists():
        print(f"Found {filename}: {path.stat().st_size / (1024 ** 2):.2f} MB")
    else:
        missing.append(str(path))

if missing:
    raise FileNotFoundError("Missing processed files:\n" + "\n".join(missing))

print("All required processed files found.")

Found train_interactions.parquet: 134.36 MB
Found val_interactions.parquet: 28.11 MB
Found test_interactions.parquet: 28.72 MB
Found train_interactions_aggregated.parquet: 129.63 MB
Found articles_processed.parquet: 6.66 MB
Found article_id_map.parquet: 1.20 MB
Found customer_id_map.parquet: 84.84 MB
Found sample_submission_processed.parquet: 84.86 MB
All required processed files found.


## 3. Configuration

In [6]:
TOP_K_VALUES = [12, 20]
PRIMARY_K = 20
RANDOM_SEED = 42

# Keep False for the first successful run. Set True later if you want to compare multiple ALS configurations.
RUN_ALS_GRID = False

ALS_CONFIGS = [
    {
        "model_name": "als_f64_reg0.05_it15_alpha40",
        "factors": 64,
        "regularization": 0.05,
        "iterations": 15,
        "alpha": 40.0,
    },
    {
        "model_name": "als_f128_reg0.05_it15_alpha40",
        "factors": 128,
        "regularization": 0.05,
        "iterations": 15,
        "alpha": 40.0,
    },
]

if not RUN_ALS_GRID:
    ALS_CONFIGS = ALS_CONFIGS[:1]

# Full evaluation is best for final reporting. If Kaggle runtime is too slow, set this to an integer like 50,000.
MAX_EVAL_USERS_PER_SPLIT = None

# Batch size for model.recommend. Reduce this if memory is tight.
RECOMMEND_BATCH_SIZE = 4096

N_DEMO_USERS = 1000
GENERATE_FULL_SAMPLE_SUBMISSION = False

np.random.seed(RANDOM_SEED)

print("ALS configs:")
for config in ALS_CONFIGS:
    print(config)
print("MAX_EVAL_USERS_PER_SPLIT:", MAX_EVAL_USERS_PER_SPLIT)

ALS configs:
{'model_name': 'als_f64_reg0.05_it15_alpha40', 'factors': 64, 'regularization': 0.05, 'iterations': 15, 'alpha': 40.0}
MAX_EVAL_USERS_PER_SPLIT: None


## 4. Utility functions

In [7]:
def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def print_df_info(df: pd.DataFrame, name: str, show_head: bool = True) -> None:
    print()
    print(name)
    print("-" * 90)
    print("Shape:", df.shape)
    print(f"Memory usage: {memory_usage_mb(df):.2f} MB")
    if show_head:
        display(df.head())


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print(f"Saved: {path}")


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, compression="snappy")
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")


def save_pickle(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with open(path, "wb") as f:
            pickle.dump(obj, f)
        print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")
    except Exception as e:
        print(f"Warning: could not pickle object to {path}: {repr(e)}")
        print("The factor arrays are still saved separately and can be used for the demo/similarity modules.")

In [8]:
def apk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen_recommended = set()

    for rank, item in enumerate(recommended[:k], start=1):
        if item in seen_recommended:
            continue
        seen_recommended.add(item)
        if item in actual:
            hits += 1
            score += hits / rank

    return score / min(len(actual), k)


def ndcgk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0

    dcg = 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(actual), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def precisionk(recommended: List[int], actual: set, k: int) -> float:
    if k <= 0:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / k


def recallk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / len(actual)


def mrrk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            return 1.0 / rank
    return 0.0


def hitratek(recommended: List[int], actual: set, k: int) -> float:
    return float(any(item in actual for item in recommended[:k]))

In [9]:
def build_user_item_sets(df: pd.DataFrame) -> Dict[int, set]:
    grouped = df.groupby("customer_idx", sort=False)["article_idx"].agg(
        lambda values: set(values.to_numpy(dtype=np.int32))
    )
    return grouped.to_dict()


def recommend_from_ranked_items(
    ranked_items: np.ndarray,
    seen_items: Optional[set],
    k: int,
    n_candidates: int = 5000,
) -> List[int]:
    if seen_items is None or len(seen_items) == 0:
        return ranked_items[:k].astype(int).tolist()

    recommendations = []
    scan_limit = min(len(ranked_items), n_candidates)

    for item in ranked_items[:scan_limit]:
        item_int = int(item)
        if item_int not in seen_items:
            recommendations.append(item_int)
            if len(recommendations) == k:
                break

    if len(recommendations) < k:
        for item in ranked_items[scan_limit:]:
            item_int = int(item)
            if item_int not in seen_items:
                recommendations.append(item_int)
                if len(recommendations) == k:
                    break

    return recommendations


def update_metric_sums(
    recommended: List[int],
    actual: set,
    k_values: List[int],
    sums: Dict[int, Dict[str, float]],
    unique_recommended_items_by_k: Dict[int, set],
) -> None:
    for k in k_values:
        top_k_recommended = recommended[:k]
        unique_recommended_items_by_k[k].update(top_k_recommended)
        sums[k]["precision"] += precisionk(recommended, actual, k)
        sums[k]["recall"] += recallk(recommended, actual, k)
        sums[k]["map"] += apk(recommended, actual, k)
        sums[k]["ndcg"] += ndcgk(recommended, actual, k)
        sums[k]["mrr"] += mrrk(recommended, actual, k)
        sums[k]["hitrate"] += hitratek(recommended, actual, k)

## 5. Load processed data

In [10]:
train_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

val_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "val_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

test_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "test_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

train_interactions_aggregated = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions_aggregated.parquet",
    columns=["customer_idx", "article_idx", "purchase_count", "implicit_weight"],
)

article_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "article_id_map.parquet")
customer_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "customer_id_map.parquet")
articles = pd.read_parquet(PROCESSED_DATA_DIR / "articles_processed.parquet")
sample_submission = pd.read_parquet(PROCESSED_DATA_DIR / "sample_submission_processed.parquet")

for df in [train_interactions, val_interactions, test_interactions]:
    df["t_dat"] = pd.to_datetime(df["t_dat"])
    df["customer_idx"] = df["customer_idx"].astype("int32")
    df["article_idx"] = df["article_idx"].astype("int32")

train_interactions_aggregated["customer_idx"] = train_interactions_aggregated["customer_idx"].astype("int32")
train_interactions_aggregated["article_idx"] = train_interactions_aggregated["article_idx"].astype("int32")
train_interactions_aggregated["implicit_weight"] = train_interactions_aggregated["implicit_weight"].astype("float32")

print_df_info(train_interactions, "Train interactions")
print_df_info(train_interactions_aggregated, "Train aggregated interactions")
print_df_info(val_interactions, "Validation interactions")
print_df_info(test_interactions, "Test interactions")
print_df_info(article_id_map, "Article ID map")
print_df_info(customer_id_map, "Customer ID map")


Train interactions
------------------------------------------------------------------------------------------
Shape: (22235277, 6)
Memory usage: 530.13 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,weight
0,2018-09-20,2,40179,0.050831,2,1.0
1,2018-09-20,7,46302,0.016932,2,1.0
2,2018-09-20,7,6386,0.020322,2,1.0
3,2018-09-20,198,47416,0.030492,1,1.0
4,2018-09-20,198,5944,0.053373,1,1.0



Train aggregated interactions
------------------------------------------------------------------------------------------
Shape: (19106319, 4)
Memory usage: 291.54 MB


,customer_idx,article_idx,purchase_count,implicit_weight
0,0,99,1.0,1.693147
1,0,16003,2.0,2.098612
2,0,23996,1.0,1.693147
3,0,29516,1.0,1.693147
4,0,30327,1.0,1.693147



Validation interactions
------------------------------------------------------------------------------------------
Shape: (4760953, 6)
Memory usage: 113.51 MB


...

,t_dat,customer_idx,article_idx,price,sales_channel_id,weight
0,2020-06-09,38,95122,0.033881,2,1.0
1,2020-06-09,38,95122,0.033881,2,1.0
2,2020-06-09,38,87885,0.025407,1,1.0
3,2020-06-09,38,94629,0.005068,1,1.0
4,2020-06-09,38,92358,0.042356,2,1.0



Article ID map
------------------------------------------------------------------------------------------
Shape: (105542, 2)
Memory usage: 6.34 MB


,article_id,article_idx
0,0108775015,0
1,0108775044,1
2,0108775051,2
3,0110065001,3
4,0110065002,4



Customer ID map
------------------------------------------------------------------------------------------
Shape: (1371980, 2)
Memory usage: 153.09 MB


,customer_id,customer_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,3
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,4


In [11]:
assert train_interactions["t_dat"].max() < val_interactions["t_dat"].min(), "Train/validation split overlap detected."
assert val_interactions["t_dat"].max() < test_interactions["t_dat"].min(), "Validation/test split overlap detected."

n_users = int(customer_id_map["customer_idx"].max()) + 1
n_items = int(article_id_map["article_idx"].max()) + 1

als_dataset_audit = {
    "n_users_from_mapping": n_users,
    "n_items_from_mapping": n_items,
    "train_events": int(len(train_interactions)),
    "train_aggregated_pairs": int(len(train_interactions_aggregated)),
    "validation_events": int(len(val_interactions)),
    "test_events": int(len(test_interactions)),
    "train_unique_users": int(train_interactions["customer_idx"].nunique()),
    "train_unique_items": int(train_interactions["article_idx"].nunique()),
    "validation_unique_users": int(val_interactions["customer_idx"].nunique()),
    "test_unique_users": int(test_interactions["customer_idx"].nunique()),
}

print(json.dumps(als_dataset_audit, indent=2))
save_json(als_dataset_audit, ALS_ARTIFACTS_DIR / "als_dataset_audit.json")

{
  "n_users_from_mapping": 1371980,
  "n_items_from_mapping": 105542,
  "train_events": 22235277,
  "train_aggregated_pairs": 19106319,
  "validation_events": 4760953,
  "test_events": 4792094,
  "train_unique_users": 1150557,
  "train_unique_items": 83275,
  "validation_unique_users": 574129,
  "test_unique_users": 578785
}
Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_dataset_audit.json


## 6. Build sparse user-item matrix

In [12]:
row_indices = train_interactions_aggregated["customer_idx"].to_numpy(dtype=np.int32)
col_indices = train_interactions_aggregated["article_idx"].to_numpy(dtype=np.int32)
values = train_interactions_aggregated["implicit_weight"].to_numpy(dtype=np.float32)

user_items = csr_matrix(
    (values, (row_indices, col_indices)),
    shape=(n_users, n_items),
    dtype=np.float32,
)

user_items.eliminate_zeros()

print("user_items shape:", user_items.shape)
print("user_items non-zero entries:", user_items.nnz)
print("density percent:", user_items.nnz / (user_items.shape[0] * user_items.shape[1]) * 100)

save_npz(ALS_ARTIFACTS_DIR / "train_user_items_matrix.npz", user_items)
print("Saved sparse matrix:", ALS_ARTIFACTS_DIR / "train_user_items_matrix.npz")

user_items shape: (1371980, 105542)
user_items non-zero entries: 19106319
density percent: 0.013194833799069671
Saved sparse matrix: /kaggle/working/hm-recommender/artifacts/als_recommender/train_user_items_matrix.npz


## 7. Popularity fallback and target sets

In [13]:
def build_global_popularity_from_train(train_df: pd.DataFrame) -> np.ndarray:
    scores = (
        train_df
        .groupby("article_idx", as_index=False)
        .agg(
            popularity_score=("weight", "sum"),
            purchase_count=("article_idx", "size"),
            unique_customers=("customer_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
        )
        .sort_values(
            ["popularity_score", "unique_customers", "last_train_purchase_date", "article_idx"],
            ascending=[False, False, False, True],
        )
    )
    return scores["article_idx"].to_numpy(dtype=np.int32)


def find_baseline_global_scores_path() -> Optional[Path]:
    expected_file = "global_popularity_scores.parquet"

    candidate_paths = [
        BASELINE_ARTIFACTS_DIR / expected_file,
    ]

    if Path("/kaggle/input").exists():
        candidate_paths.extend(sorted(Path("/kaggle/input").rglob(expected_file)))

    for path in candidate_paths:
        if path.exists():
            return path

    return None


baseline_global_scores_path = find_baseline_global_scores_path()

if baseline_global_scores_path is not None:
    global_popularity_scores = pd.read_parquet(baseline_global_scores_path)
    global_ranked_items = global_popularity_scores["article_idx"].to_numpy(dtype=np.int32)
    print("Loaded global popularity fallback from:", baseline_global_scores_path)
else:
    global_ranked_items = build_global_popularity_from_train(train_interactions)
    print("Built global popularity fallback from train interactions because baseline artifact was not found.")

print("Fallback ranked items:", len(global_ranked_items))

print("Building train seen and validation/test target dictionaries...")
started = time.time()
train_seen_by_user = build_user_item_sets(train_interactions)
val_actual_by_user = build_user_item_sets(val_interactions)
test_actual_by_user = build_user_item_sets(test_interactions)
print(f"Built dictionaries in {time.time() - started:.2f} seconds.")

split_user_audit = {
    "train_seen_users": len(train_seen_by_user),
    "val_target_users": len(val_actual_by_user),
    "test_target_users": len(test_actual_by_user),
    "val_warm_users": sum(1 for u in val_actual_by_user if u in train_seen_by_user),
    "test_warm_users": sum(1 for u in test_actual_by_user if u in train_seen_by_user),
    "val_cold_users": sum(1 for u in val_actual_by_user if u not in train_seen_by_user),
    "test_cold_users": sum(1 for u in test_actual_by_user if u not in train_seen_by_user),
}

print(json.dumps(split_user_audit, indent=2))
save_json(split_user_audit, ALS_ARTIFACTS_DIR / "als_split_user_audit.json")

Loaded global popularity fallback from: /kaggle/input/datasets/albaky7/hm-popularity-baseline/hm-recommender/artifacts/popularity_baseline/global_popularity_scores.parquet
Fallback ranked items: 83275
Building train seen and validation/test target dictionaries...
Built dictionaries in 58.57 seconds.
{
  "train_seen_users": 1150557,
  "val_target_users": 574129,
  "test_target_users": 578785,
  "val_warm_users": 450725,
  "test_warm_users": 456034,
  "val_cold_users": 123404,
  "test_cold_users": 122751
}
Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_split_user_audit.json


## 8. ALS recommendation and evaluation helpers

In [14]:
def sample_eval_users(user_ids: List[int], max_users: Optional[int], seed: int = RANDOM_SEED) -> List[int]:
    if max_users is None or len(user_ids) <= max_users:
        return list(user_ids)
    rng = np.random.default_rng(seed)
    return rng.choice(np.array(user_ids, dtype=np.int32), size=max_users, replace=False).astype(int).tolist()


def batch_recommend_als(
    model: AlternatingLeastSquares,
    user_items_matrix: csr_matrix,
    user_ids: List[int],
    k: int,
    batch_size: int = 4096,
) -> Iterable[Tuple[int, List[int]]]:
    """Yield (user_id, recommendations) using implicit's batch API when available."""
    for start in range(0, len(user_ids), batch_size):
        batch_users = np.array(user_ids[start:start + batch_size], dtype=np.int32)
        if len(batch_users) == 0:
            continue

        try:
            ids, scores = model.recommend(
                batch_users,
                user_items_matrix[batch_users],
                N=k,
                filter_already_liked_items=True,
                recalculate_user=False,
            )
            for user_id, rec_items in zip(batch_users, ids):
                yield int(user_id), [int(item) for item in rec_items[:k] if int(item) >= 0]
        except Exception as batch_error:
            print("Batch recommend failed once; falling back to per-user recommendation.")
            print("Batch error:", repr(batch_error))
            for user_id in batch_users:
                rec_items, scores = model.recommend(
                    int(user_id),
                    user_items_matrix[int(user_id)],
                    N=k,
                    filter_already_liked_items=True,
                    recalculate_user=False,
                )
                yield int(user_id), [int(item) for item in rec_items[:k] if int(item) >= 0]


def evaluate_als_model(
    model: AlternatingLeastSquares,
    model_name: str,
    user_items_matrix: csr_matrix,
    train_seen_by_user: Dict[int, set],
    eval_actual_by_user: Dict[int, set],
    fallback_ranked_items: np.ndarray,
    k_values: List[int],
    eval_name: str,
    warm_start_only: bool,
    total_catalog_items: int,
    max_eval_users: Optional[int] = None,
) -> Tuple[pd.DataFrame, Dict[int, List[int]]]:
    started = time.time()
    max_k = max(k_values)

    all_eval_user_ids = list(eval_actual_by_user.keys())
    if warm_start_only:
        eval_user_ids = [u for u in all_eval_user_ids if u in train_seen_by_user]
    else:
        eval_user_ids = all_eval_user_ids

    eval_user_ids = sample_eval_users(eval_user_ids, max_eval_users)

    if not eval_user_ids:
        raise ValueError(f"No evaluation users found for {model_name}, {eval_name}, warm_start_only={warm_start_only}")

    warm_user_ids = [int(u) for u in eval_user_ids if u in train_seen_by_user]
    cold_user_ids = [int(u) for u in eval_user_ids if u not in train_seen_by_user]

    sums = {
        k: {"precision": 0.0, "recall": 0.0, "map": 0.0, "ndcg": 0.0, "mrr": 0.0, "hitrate": 0.0}
        for k in k_values
    }
    unique_recommended_items_by_k = {k: set() for k in k_values}
    sample_recommendations_by_user = {}
    scored_users = 0

    for user_id, recommended in batch_recommend_als(
        model=model,
        user_items_matrix=user_items_matrix,
        user_ids=warm_user_ids,
        k=max_k,
        batch_size=RECOMMEND_BATCH_SIZE,
    ):
        actual = eval_actual_by_user[user_id]
        if len(sample_recommendations_by_user) < 10:
            sample_recommendations_by_user[user_id] = recommended
        update_metric_sums(recommended, actual, k_values, sums, unique_recommended_items_by_k)
        scored_users += 1

    # Cold users cannot be personalized by ALS, so full evaluation uses popularity fallback.
    # In warm-start-only mode there should be no cold users.
    if not warm_start_only:
        for user_id in cold_user_ids:
            actual = eval_actual_by_user[user_id]
            recommended = recommend_from_ranked_items(
                ranked_items=fallback_ranked_items,
                seen_items=None,
                k=max_k,
                n_candidates=5000,
            )
            if len(sample_recommendations_by_user) < 10:
                sample_recommendations_by_user[user_id] = recommended
            update_metric_sums(recommended, actual, k_values, sums, unique_recommended_items_by_k)
            scored_users += 1

    if scored_users == 0:
        raise ValueError(f"No users scored for {model_name}, {eval_name}, warm_start_only={warm_start_only}")

    rows = []
    elapsed_seconds = time.time() - started

    for k in k_values:
        rows.append(
            {
                "model_name": model_name,
                "eval_split": eval_name,
                "warm_start_only": warm_start_only,
                "uses_popularity_fallback_for_cold_users": not warm_start_only,
                "k": k,
                "n_eval_users": scored_users,
                "n_warm_users": len(warm_user_ids),
                "n_cold_users": 0 if warm_start_only else len(cold_user_ids),
                "precision_at_k": sums[k]["precision"] / scored_users,
                "recall_at_k": sums[k]["recall"] / scored_users,
                "map_at_k": sums[k]["map"] / scored_users,
                "ndcg_at_k": sums[k]["ndcg"] / scored_users,
                "mrr_at_k": sums[k]["mrr"] / scored_users,
                "hitrate_at_k": sums[k]["hitrate"] / scored_users,
                "unique_recommended_items": len(unique_recommended_items_by_k[k]),
                "catalog_coverage": len(unique_recommended_items_by_k[k]) / total_catalog_items if total_catalog_items > 0 else np.nan,
                "elapsed_seconds": round(elapsed_seconds, 2),
                "max_eval_users_per_split": max_eval_users,
            }
        )

    return pd.DataFrame(rows), sample_recommendations_by_user

## 9. Train and evaluate ALS

In [15]:
def train_als_model(config: dict, user_items_matrix: csr_matrix) -> AlternatingLeastSquares:
    print()
    print("Training ALS model:", config["model_name"])
    print(config)

    model = AlternatingLeastSquares(
        factors=int(config["factors"]),
        regularization=float(config["regularization"]),
        iterations=int(config["iterations"]),
        alpha=float(config["alpha"]),
        random_state=RANDOM_SEED,
        use_gpu=False,
        calculate_training_loss=False,
    )

    started = time.time()
    model.fit(user_items_matrix, show_progress=True)
    elapsed = time.time() - started
    print(f"Training completed in {elapsed:.2f} seconds.")

    return model


all_als_metric_frames = []
trained_models = {}
all_als_sample_recommendations = {}

total_catalog_items = int(train_interactions["article_idx"].nunique())

for config in ALS_CONFIGS:
    model_name = config["model_name"]
    model = train_als_model(config, user_items)
    trained_models[model_name] = model

    model_dir = ALS_ARTIFACTS_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    save_pickle(model, model_dir / "als_model.pkl")
    np.save(model_dir / "user_factors.npy", model.user_factors)
    np.save(model_dir / "item_factors.npy", model.item_factors)
    save_json(config, model_dir / "als_config.json")

    print("Saved factor arrays:")
    print(model_dir / "user_factors.npy")
    print(model_dir / "item_factors.npy")

    for eval_name, actual_by_user in [
        ("validation", val_actual_by_user),
        ("test", test_actual_by_user),
    ]:
        for warm_start_only in [False, True]:
            metrics_df, sample_recs = evaluate_als_model(
                model=model,
                model_name=model_name,
                user_items_matrix=user_items,
                train_seen_by_user=train_seen_by_user,
                eval_actual_by_user=actual_by_user,
                fallback_ranked_items=global_ranked_items,
                k_values=TOP_K_VALUES,
                eval_name=eval_name,
                warm_start_only=warm_start_only,
                total_catalog_items=total_catalog_items,
                max_eval_users=MAX_EVAL_USERS_PER_SPLIT,
            )
            display(metrics_df)
            all_als_metric_frames.append(metrics_df)
            sample_key = f"{model_name}_{eval_name}_{'warm' if warm_start_only else 'full'}"
            all_als_sample_recommendations[sample_key] = sample_recs

als_metrics = pd.concat(all_als_metric_frames, ignore_index=True)

metric_cols = [
    "precision_at_k",
    "recall_at_k",
    "map_at_k",
    "ndcg_at_k",
    "mrr_at_k",
    "hitrate_at_k",
    "catalog_coverage",
]
als_metrics[metric_cols] = als_metrics[metric_cols].round(6)

als_metrics_sorted = als_metrics.sort_values(
    ["eval_split", "warm_start_only", "k", "recall_at_k", "ndcg_at_k", "mrr_at_k"],
    ascending=[True, True, True, False, False, False],
).reset_index(drop=True)

display(als_metrics_sorted)
save_csv(als_metrics, ALS_ARTIFACTS_DIR / "als_metrics.csv")


Training ALS model: als_f64_reg0.05_it15_alpha40
{'model_name': 'als_f64_reg0.05_it15_alpha40', 'factors': 64, 'regularization': 0.05, 'iterations': 15, 'alpha': 40.0}


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

Training completed in 130.89 seconds.
Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_f64_reg0.05_it15_alpha40/als_model.pkl | 360.72 MB
Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_f64_reg0.05_it15_alpha40/als_config.json
Saved factor arrays:
/kaggle/working/hm-recommender/artifacts/als_recommender/als_f64_reg0.05_it15_alpha40/user_factors.npy
/kaggle/working/hm-recommender/artifacts/als_recommender/als_f64_reg0.05_it15_alpha40/item_factors.npy


,model_name,eval_split,warm_start_only,uses_popularity_fallback_for_cold_users,k,n_eval_users,n_warm_users,n_cold_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users_per_split
0,als_f64_reg0.05_it15_alpha40,validation,False,True,12,574129,450725,123404,0.004493,0.009983,0.004141,0.008657,0.017765,0.048224,7359,0.088370,249.5,None
1,als_f64_reg0.05_it15_alpha40,validation,False,True,20,574129,450725,123404,0.003923,0.014411,0.004297,0.010075,0.019013,0.068293,8692,0.104377,249.5,None


,model_name,eval_split,warm_start_only,uses_popularity_fallback_for_cold_users,k,n_eval_users,n_warm_users,n_cold_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users_per_split
0,als_f64_reg0.05_it15_alpha40,validation,True,False,12,450725,450725,0,0.004780,0.009703,0.003931,0.008637,0.018494,0.050933,7359,0.088370,246.12,None
1,als_f64_reg0.05_it15_alpha40,validation,True,False,20,450725,450725,0,0.004018,0.013200,0.004000,0.009720,0.019642,0.069293,8692,0.104377,246.12,None


,model_name,eval_split,warm_start_only,uses_popularity_fallback_for_cold_users,k,n_eval_users,n_warm_users,n_cold_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users_per_split
0,als_f64_reg0.05_it15_alpha40,test,False,True,12,578785,456034,122751,0.003257,0.007103,0.002829,0.006136,0.012927,0.035884,7367,0.088466,251.58,None
1,als_f64_reg0.05_it15_alpha40,test,False,True,20,578785,456034,122751,0.002799,0.010045,0.002924,0.007081,0.013806,0.049953,8682,0.104257,251.58,None


,model_name,eval_split,warm_start_only,uses_popularity_fallback_for_cold_users,k,n_eval_users,n_warm_users,n_cold_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users_per_split
0,als_f64_reg0.05_it15_alpha40,test,True,False,12,456034,456034,0,0.003422,0.006934,0.002773,0.006173,0.013380,0.037383,7367,0.088466,249.87,None
1,als_f64_reg0.05_it15_alpha40,test,True,False,20,456034,456034,0,0.002859,0.009449,0.002824,0.006953,0.014223,0.050801,8682,0.104257,249.87,None


,model_name,eval_split,warm_start_only,uses_popularity_fallback_for_cold_users,k,n_eval_users,n_warm_users,n_cold_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users_per_split
0,als_f64_reg0.05_it15_alpha40,test,False,True,12,578785,456034,122751,0.003257,0.007103,0.002829,0.006136,0.012927,0.035884,7367,0.088466,251.58,None
1,als_f64_reg0.05_it15_alpha40,test,False,True,20,578785,456034,122751,0.002799,0.010045,0.002924,0.007081,0.013806,0.049953,8682,0.104257,251.58,None
2,als_f64_reg0.05_it15_alpha40,test,True,False,12,456034,456034,0,0.003422,0.006934,0.002773,0.006173,0.013380,0.037383,7367,0.088466,249.87,None
3,als_f64_reg0.05_it15_alpha40,test,True,False,20,456034,456034,0,0.002859,0.009449,0.002824,0.006953,0.014223,0.050801,8682,0.104257,249.87,None
4,als_f64_reg0.05_it15_alpha40,validation,False,True,12,574129,450725,123404,0.004493,0.009983,0.004141,0.008657,0.017765,0.048224,7359,0.088370,249.50,None
5,als_f64_reg0.05_it15_alpha40,validation,False,True,20,574129,450725,123404,0.003923,0.014411,0.004297,0.010075,0.019013,0.068293,8692,0.104377,249.50,None
6,als_f64_reg0.05_it15_alpha40,validation,True,False,12,450725,450725,0,0.004780,0.009703,0.003931,0.008637,0.018494,0.050933,7359,0.088370,246.12,None
7,als_f64_reg0.05_it15_alpha40,validation,True,False,20,450725,450725,0,0.004018,0.013200,0.004000,0.009720,0.019642,0.069293,8692,0.104377,246.12,None


Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_metrics.csv | 0.00 MB


## 10. Select best ALS model

In [16]:
primary_als_leaderboard = (
    als_metrics[
        (als_metrics["eval_split"] == "validation")
        & (als_metrics["warm_start_only"] == True)
        & (als_metrics["k"] == PRIMARY_K)
    ]
    .sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k"], ascending=False)
    .reset_index(drop=True)
)

display(primary_als_leaderboard)

best_als_model_name = str(primary_als_leaderboard.loc[0, "model_name"])
best_als_model = trained_models[best_als_model_name]

best_als_info = {
    "best_als_model_name": best_als_model_name,
    "selection_split": "validation",
    "selection_mode": "warm_start_only",
    "selection_k": PRIMARY_K,
    "selection_metric": "recall_at_k",
    "best_validation_metrics": primary_als_leaderboard.iloc[0].to_dict(),
    "artifact_dir": str(ALS_ARTIFACTS_DIR / best_als_model_name),
}

save_json(best_als_info, ALS_ARTIFACTS_DIR / "best_als_model_info.json")
print("Best ALS model:", best_als_model_name)

,model_name,eval_split,warm_start_only,uses_popularity_fallback_for_cold_users,k,n_eval_users,n_warm_users,n_cold_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users_per_split
0,als_f64_reg0.05_it15_alpha40,validation,True,False,20,450725,450725,0,0.004018,0.0132,0.004,0.00972,0.019642,0.069293,8692,0.104377,246.12,None


Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/best_als_model_info.json
Best ALS model: als_f64_reg0.05_it15_alpha40


## 11. Demo recommendations

In [17]:
def recommend_users_with_als_and_fallback(
    user_ids: Iterable[int],
    model: AlternatingLeastSquares,
    user_items_matrix: csr_matrix,
    train_seen_by_user: Dict[int, set],
    fallback_ranked_items: np.ndarray,
    k: int,
) -> Dict[int, List[int]]:
    user_ids = [int(u) for u in user_ids]
    warm_user_ids = [u for u in user_ids if u in train_seen_by_user]
    cold_user_ids = [u for u in user_ids if u not in train_seen_by_user]

    recs_by_user = {}

    for user_id, recs in batch_recommend_als(
        model=model,
        user_items_matrix=user_items_matrix,
        user_ids=warm_user_ids,
        k=k,
        batch_size=RECOMMEND_BATCH_SIZE,
    ):
        recs_by_user[user_id] = recs

    for user_id in cold_user_ids:
        recs_by_user[user_id] = recommend_from_ranked_items(
            ranked_items=fallback_ranked_items,
            seen_items=None,
            k=k,
            n_candidates=5000,
        )

    return recs_by_user


def build_als_recommendations_for_users(
    user_ids: Iterable[int],
    model: AlternatingLeastSquares,
    user_items_matrix: csr_matrix,
    train_seen_by_user: Dict[int, set],
    fallback_ranked_items: np.ndarray,
    k: int,
    model_name: str,
) -> pd.DataFrame:
    recs_by_user = recommend_users_with_als_and_fallback(
        user_ids=user_ids,
        model=model,
        user_items_matrix=user_items_matrix,
        train_seen_by_user=train_seen_by_user,
        fallback_ranked_items=fallback_ranked_items,
        k=k,
    )

    rows = []
    for user_id, recs in recs_by_user.items():
        used_fallback = user_id not in train_seen_by_user
        for rank, article_idx in enumerate(recs, start=1):
            rows.append(
                {
                    "customer_idx": int(user_id),
                    "article_idx": int(article_idx),
                    "rank": rank,
                    "model_name": model_name,
                    "used_popularity_fallback": used_fallback,
                }
            )

    return pd.DataFrame(rows)


demo_user_ids = (
    sample_submission["customer_idx"]
    .dropna()
    .astype("int32")
    .head(N_DEMO_USERS)
    .to_numpy()
)

demo_als_recommendations = build_als_recommendations_for_users(
    user_ids=demo_user_ids,
    model=best_als_model,
    user_items_matrix=user_items,
    train_seen_by_user=train_seen_by_user,
    fallback_ranked_items=global_ranked_items,
    k=PRIMARY_K,
    model_name=best_als_model_name,
)

demo_als_recommendations = demo_als_recommendations.merge(
    article_id_map[["article_idx", "article_id"]],
    on="article_idx",
    how="left",
    validate="many_to_one",
)

article_preview_cols = [
    "article_idx",
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "department_name",
    "section_name",
    "garment_group_name",
]
article_preview_cols = [col for col in article_preview_cols if col in articles.columns]

demo_als_recommendations = demo_als_recommendations.merge(
    articles[article_preview_cols],
    on=["article_idx", "article_id"],
    how="left",
    validate="many_to_one",
)

print_df_info(demo_als_recommendations, "Demo ALS recommendations")
save_parquet(demo_als_recommendations, ALS_ARTIFACTS_DIR / "demo_als_recommendations.parquet")


Demo ALS recommendations
------------------------------------------------------------------------------------------
Shape: (20000, 13)
Memory usage: 11.29 MB


,customer_idx,article_idx,rank,model_name,used_popularity_fallback,article_id,prod_name,product_type_name,product_group_name,colour_group_name,department_name,section_name,garment_group_name
0,0,15988,1,als_f64_reg0.05_it15_alpha40,False,0568597006,Hayes slim trouser,Trousers,Garment Lower body,Black,Trouser,Womens Tailoring,Trousers
1,0,67522,2,als_f64_reg0.05_it15_alpha40,False,0751471001,Pluto RW slacks (1),Trousers,Garment Lower body,Black,Trouser,Womens Everyday Collection,Trousers
2,0,6643,3,als_f64_reg0.05_it15_alpha40,False,0507909003,Rebecca or Delphine shirt,Shirt,Garment Upper body,Black,Blouse,Womens Tailoring,Blouses
3,0,42626,4,als_f64_reg0.05_it15_alpha40,False,0673677002,Henry polo. (1),Sweater,Garment Upper body,Black,Knitwear,Womens Tailoring,Knitwear
4,0,67943,5,als_f64_reg0.05_it15_alpha40,False,0752814003,Milk RW slack,Trousers,Garment Lower body,Black,Trouser,Womens Everyday Collection,Trousers


Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/demo_als_recommendations.parquet | 0.22 MB


## 12. Optional Kaggle-style prediction file

In [18]:
def build_kaggle_style_als_predictions(
    users_df: pd.DataFrame,
    model: AlternatingLeastSquares,
    user_items_matrix: csr_matrix,
    train_seen_by_user: Dict[int, set],
    fallback_ranked_items: np.ndarray,
    k: int = 12,
) -> pd.DataFrame:
    article_idx_to_article_id = dict(
        zip(article_id_map["article_idx"].astype(int), article_id_map["article_id"].astype(str))
    )

    user_ids = users_df["customer_idx"].astype("int32").tolist()
    recs_by_user = recommend_users_with_als_and_fallback(
        user_ids=user_ids,
        model=model,
        user_items_matrix=user_items_matrix,
        train_seen_by_user=train_seen_by_user,
        fallback_ranked_items=fallback_ranked_items,
        k=k,
    )

    rows = []
    for row in users_df[["customer_id", "customer_idx"]].itertuples(index=False):
        customer_id = row.customer_id
        customer_idx = int(row.customer_idx)
        recs = recs_by_user[customer_idx]
        prediction = " ".join(article_idx_to_article_id[int(item)] for item in recs)
        rows.append({"customer_id": customer_id, "prediction": prediction})

    return pd.DataFrame(rows)


if GENERATE_FULL_SAMPLE_SUBMISSION:
    started = time.time()
    full_als_prediction_df = build_kaggle_style_als_predictions(
        users_df=sample_submission.dropna(subset=["customer_idx"]),
        model=best_als_model,
        user_items_matrix=user_items,
        train_seen_by_user=train_seen_by_user,
        fallback_ranked_items=global_ranked_items,
        k=12,
    )
    save_csv(full_als_prediction_df, ALS_ARTIFACTS_DIR / "als_submission_like.csv")
    print(f"Generated full ALS sample-submission-style file in {time.time() - started:.2f} seconds.")
else:
    small_als_prediction_df = build_kaggle_style_als_predictions(
        users_df=sample_submission.dropna(subset=["customer_idx"]).head(N_DEMO_USERS),
        model=best_als_model,
        user_items_matrix=user_items,
        train_seen_by_user=train_seen_by_user,
        fallback_ranked_items=global_ranked_items,
        k=12,
    )
    display(small_als_prediction_df.head())
    save_csv(small_als_prediction_df, ALS_ARTIFACTS_DIR / "als_submission_like_demo.csv")
    print("Skipped full submission file. Set GENERATE_FULL_SAMPLE_SUBMISSION = True if needed.")

,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568597006 0751471001 0507909003 0673677002 07...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0599580017 0603584018 0795440001 0688537004 06...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0458543001 0609719001 0699080001 0699081001 05...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0730683001 0720125001 0740498001 0738133005 06...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0692721005 0578476001 0586896039 0589599035 05...


Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_submission_like_demo.csv | 0.19 MB
Skipped full submission file. Set GENERATE_FULL_SAMPLE_SUBMISSION = True if needed.


## 13. ALS embedding sanity checks for future similarity modules

In [19]:
# These small outputs confirm that the ALS embeddings are available for Milestone 4.
# They are not the full similarity module yet, but they prove that similar_items and similar_users can be built next.

sanity_outputs = {}

try:
    first_item = int(train_interactions["article_idx"].value_counts().index[0])
    similar_item_ids, similar_item_scores = best_als_model.similar_items(first_item, N=10)
    similar_items_demo = pd.DataFrame(
        {
            "query_article_idx": first_item,
            "similar_article_idx": similar_item_ids.astype(int),
            "similarity_score": similar_item_scores.astype(float),
            "rank": np.arange(1, len(similar_item_ids) + 1),
        }
    )
    similar_items_demo = similar_items_demo.merge(
        article_id_map.rename(columns={"article_idx": "similar_article_idx", "article_id": "similar_article_id"})[
            ["similar_article_idx", "similar_article_id"]
        ],
        on="similar_article_idx",
        how="left",
    )
    display(similar_items_demo)
    save_parquet(similar_items_demo, ALS_ARTIFACTS_DIR / "als_similar_items_demo.parquet")
    sanity_outputs["similar_items_demo_saved"] = True
except Exception as e:
    print("Could not create similar-items sanity output:", repr(e))
    sanity_outputs["similar_items_demo_saved"] = False

try:
    first_user = int(train_interactions["customer_idx"].value_counts().index[0])
    similar_user_ids, similar_user_scores = best_als_model.similar_users(first_user, N=10)
    similar_users_demo = pd.DataFrame(
        {
            "query_customer_idx": first_user,
            "similar_customer_idx": similar_user_ids.astype(int),
            "similarity_score": similar_user_scores.astype(float),
            "rank": np.arange(1, len(similar_user_ids) + 1),
        }
    )
    display(similar_users_demo)
    save_parquet(similar_users_demo, ALS_ARTIFACTS_DIR / "als_similar_users_demo.parquet")
    sanity_outputs["similar_users_demo_saved"] = True
except Exception as e:
    print("Could not create similar-users sanity output:", repr(e))
    sanity_outputs["similar_users_demo_saved"] = False

save_json(sanity_outputs, ALS_ARTIFACTS_DIR / "als_embedding_sanity_outputs.json")

,query_article_idx,similar_article_idx,similarity_score,rank,similar_article_id
0,53892,53892,1.000000,1,0706016001
1,53892,53893,0.992402,2,0706016002
2,53892,53894,0.984559,3,0706016003
3,53892,53902,0.980372,4,0706016015
4,53892,53896,0.976641,5,0706016006
5,53892,53897,0.938359,6,0706016007
6,53892,53904,0.927766,7,0706016019
7,53892,53895,0.924292,8,0706016004
8,53892,10285,0.909947,9,0539723001
9,53892,10289,0.896375,10,0539723005


Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_similar_items_demo.parquet | 0.00 MB


,query_customer_idx,similar_customer_idx,similarity_score,rank
0,1018839,1018839,1.000000,1
1,1018839,582904,0.608679,2
2,1018839,527242,0.603683,3
3,1018839,865723,0.595757,4
4,1018839,977462,0.582447,5
5,1018839,952848,0.571865,6
6,1018839,529459,0.571429,7
7,1018839,15068,0.570473,8
8,1018839,1107578,0.564771,9
9,1018839,70577,0.559610,10


Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_similar_users_demo.parquet | 0.00 MB
Saved: /kaggle/working/hm-recommender/artifacts/als_recommender/als_embedding_sanity_outputs.json


## 14. Final summary

In [20]:
final_summary = {
    "best_als_model_name": best_als_model_name,
    "artifact_dir": str(ALS_ARTIFACTS_DIR),
    "model_artifact_dir": str(ALS_ARTIFACTS_DIR / best_als_model_name),
    "metrics_file": str(ALS_ARTIFACTS_DIR / "als_metrics.csv"),
    "demo_recommendations_file": str(ALS_ARTIFACTS_DIR / "demo_als_recommendations.parquet"),
    "saved_files": sorted([p.name for p in ALS_ARTIFACTS_DIR.glob("*")]),
}

print(json.dumps(final_summary, indent=2))

print()
print("ALS recommender completed successfully.")
print("Next step: 05 similarity modules using metadata + ALS embeddings.")

{
  "best_als_model_name": "als_f64_reg0.05_it15_alpha40",
  "artifact_dir": "/kaggle/working/hm-recommender/artifacts/als_recommender",
  "model_artifact_dir": "/kaggle/working/hm-recommender/artifacts/als_recommender/als_f64_reg0.05_it15_alpha40",
  "metrics_file": "/kaggle/working/hm-recommender/artifacts/als_recommender/als_metrics.csv",
  "demo_recommendations_file": "/kaggle/working/hm-recommender/artifacts/als_recommender/demo_als_recommendations.parquet",
  "saved_files": [
    "als_dataset_audit.json",
    "als_embedding_sanity_outputs.json",
    "als_f64_reg0.05_it15_alpha40",
    "als_metrics.csv",
    "als_similar_items_demo.parquet",
    "als_similar_users_demo.parquet",
    "als_split_user_audit.json",
    "als_submission_like_demo.csv",
    "best_als_model_info.json",
    "demo_als_recommendations.parquet",
    "train_user_items_matrix.npz"
  ]
}

ALS recommender completed successfully.
Next step: 05 similarity modules using metadata + ALS embeddings.
